In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

In [15]:
def fetch_creditcard(X_y_split: bool = False
                     ) -> pd.DataFrame | tuple[pd.DataFrame, pd.Series]:
    response = requests.get("http://pipeline:8000/dataset/creditcard-churn", params={"X_y_split": X_y_split})
    payload = response.json()
    
    if X_y_split:
        X = pd.DataFrame(payload["x"], index=payload["index"])
        y = pd.Series(payload["y"], index=payload["index"])
        return X, y
    
    df = pd.DataFrame(payload["data"], index=payload["index"])
    
    return df

In [16]:
# 데이터 로드
bank_df = fetch_creditcard()
print(bank_df.columns)

Index(['creditcard_churn_id', 'churn', 'age', 'gender', 'dependents',
       'education_id', 'marital_id', 'income_id', 'card_type_id',
       'relationship_months', 'product_count', 'inactive_months',
       'contact_count', 'credit_limit', 'revolving_balance',
       'available_credit', 'amount_change', 'count_change',
       'transaction_amount', 'transaction_count', 'utilization_ratio'],
      dtype='object')


In [17]:
# 목표 데이터 분리
churn_df = bank_df["churn"]
print(churn_df.value_counts())

churn
0    8561
1    1666
Name: count, dtype: int64


In [18]:
# feature 데이터 분리
feature_df = bank_df[[k for k in bank_df.columns if k != "churn"]]
print(feature_df.columns)

Index(['creditcard_churn_id', 'age', 'gender', 'dependents', 'education_id',
       'marital_id', 'income_id', 'card_type_id', 'relationship_months',
       'product_count', 'inactive_months', 'contact_count', 'credit_limit',
       'revolving_balance', 'available_credit', 'amount_change',
       'count_change', 'transaction_amount', 'transaction_count',
       'utilization_ratio'],
      dtype='object')


In [19]:
from sklearn.model_selection import train_test_split

# train - test split을 활용해서 학습/테스트 데이터 분리
train_X, test_X, train_y, test_y = train_test_split(
    feature_df,
    churn_df,
    test_size=0.2,
    stratify=churn_df,
    random_state=42
)

print(train_X.describe())

       creditcard_churn_id          age       gender   dependents  \
count          8181.000000  8181.000000  8181.000000  8181.000000   
mean           5130.666544    46.315365     0.468280     2.349102   
std            2964.161172     8.064247     0.499023     1.299942   
min               1.000000    25.000000     0.000000     0.000000   
25%            2559.000000    41.000000     0.000000     1.000000   
50%            5122.000000    46.000000     0.000000     2.000000   
75%            7695.000000    52.000000     1.000000     3.000000   
max           10254.000000    73.000000     1.000000     5.000000   

       education_id   marital_id    income_id  card_type_id  \
count   8181.000000  8181.000000  8181.000000   8181.000000   
mean       3.654810     1.842073     2.745508      1.104633   
std        1.910247     0.863124     1.712108      0.410503   
min        1.000000     1.000000     1.000000      1.000000   
25%        2.000000     1.000000     1.000000      1.000000   


In [20]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.multiclass import type_of_target


def evaluate_predictions(y_true, y_pred, y_proba, classes=None, average="macro", dataset_name="Dataset"):
    target_type = type_of_target(y_true)

    result = {
        "dataset": dataset_name,
        "target_type": target_type,
        "accuracy": accuracy_score(y_true, y_pred),
    }

    # y_proba shape 보정
    y_proba = np.asarray(y_proba)

    if target_type == "binary":
        result["recall"] = recall_score(y_true, y_pred)
        result["f1"] = f1_score(y_true, y_pred)

        # (n_samples, 2) 이면 positive class 확률만 사용
        if y_proba.ndim == 2 and y_proba.shape[1] == 2:
            y_score = y_proba[:, 1]
        # 이미 (n_samples,) 형태면 그대로 사용
        elif y_proba.ndim == 1:
            y_score = y_proba
        else:
            print(f"Binary Classification Shape Error: y_proba.shape = {y_proba.shape}")
            return {}

        result["roc_auc"] = roc_auc_score(y_true, y_score)
        result["pr_auc"] = average_precision_score(y_true, y_score)

    elif target_type == "multiclass":
        result["recall"] = recall_score(y_true, y_pred, average=average)
        result["f1"] = f1_score(y_true, y_pred, average=average)

        if y_proba.ndim != 2:
            raise ValueError(f"Multiclass classification에서는 y_proba가 2차원이어야 합니다: {y_proba.shape}")

        result["roc_auc"] = roc_auc_score(
            y_true,
            y_proba,
            multi_class="ovr",
            average=average
        )

        if classes is None:
            classes = np.unique(y_true)

        y_true_bin = label_binarize(y_true, classes=classes)
        result["pr_auc"] = average_precision_score(
            y_true_bin,
            y_proba,
            average=average
        )

    else:
        raise ValueError(f"지원하지 않는 target type입니다: {target_type}")

    return result

def print_report(res):
    print(f"========== {res["dataset"]} ==========")
    print(f"target type: {res["target_type"]}")
    for k in [key for key in res if key not in ["dataset", "target_type"]]:
        print(f"{k:>11}: {res[k]:.6f}")

In [22]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_clf = HistGradientBoostingClassifier(
    random_state=42,
    max_iter=5000,
    validation_fraction=0.2,
    early_stopping=True,
    n_iter_no_change=10
)

hgb_clf.fit(train_X, train_y)

train_pred = hgb_clf.predict(train_X)
test_pred = hgb_clf.predict(test_X)

train_proba = hgb_clf.predict_proba(train_X)
test_proba = hgb_clf.predict_proba(test_X)

train_result = evaluate_predictions(
    y_true=train_y,
    y_pred=train_pred,
    y_proba=train_proba,
    classes=hgb_clf.classes_,
    dataset_name="Train"
)

test_result = evaluate_predictions(
    y_true=test_y,
    y_pred=test_pred,
    y_proba=test_proba,
    classes=hgb_clf.classes_,
    dataset_name="Test"
)

print_report(train_result)
print_report(test_result)

========== Train ==========
target type: binary
   accuracy: 0.996700
     recall: 0.987997
         f1: 0.989853
    roc_auc: 0.999469
     pr_auc: 0.997820
========== Test ==========
target type: binary
   accuracy: 0.986315
     recall: 0.939940
         f1: 0.957187
    roc_auc: 0.997851
     pr_auc: 0.989923


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(random_state=42, early_stopping=True)

# 최적의 경우를 찾고자 하는 하이퍼 파라미터들을 dictionary 형태로 준비
params = {
    'max_iter':[800, 1000, 1200],    # 약학습기 개수
    'validation_fraction':[0.1, 0.2, 0.3], # 일정 비율을 검증용 데이터로 사용
    'n_iter_no_change':[15, 20, 25], # 충분히 멈춰도 되는 지점이 오더라도 일정 횟수 더 학습하라는 의미
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='roc_auc')
grid_search.fit(train_X, train_y)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 800, 'n_iter_no_change': 25, 'validation_fraction': 0.2}
0.9974011571172138
HistGradientBoostingClassifier(early_stopping=True, max_iter=800,
                               n_iter_no_change=25, random_state=42,
                               validation_fraction=0.2)


In [25]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(random_state=42, early_stopping=True, validation_fraction=0.2)

# 최적의 경우를 찾고자 하는 하이퍼 파라미터들을 dictionary 형태로 준비
params = {
    'max_iter':[200, 500, 800],    # 약학습기 개수
    'n_iter_no_change':[25, 30, 35], # 충분히 멈춰도 되는 지점이 오더라도 일정 횟수 더 학습하라는 의미
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='roc_auc')
grid_search.fit(train_X, train_y)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 200, 'n_iter_no_change': 25}
0.9974011571172138
HistGradientBoostingClassifier(early_stopping=True, max_iter=200,
                               n_iter_no_change=25, random_state=42,
                               validation_fraction=0.2)


In [28]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=25
)

# 최적의 경우를 찾고자 하는 하이퍼 파라미터들을 dictionary 형태로 준비
params = {
    'max_iter':[100, 150, 200],    # 약학습기 개수
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='roc_auc')
grid_search.fit(train_X, train_y)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 150}
0.9974011571172138
HistGradientBoostingClassifier(early_stopping=True, max_iter=150,
                               n_iter_no_change=25, random_state=42,
                               validation_fraction=0.2)


In [29]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_clf = HistGradientBoostingClassifier(
    random_state=42,
    max_iter=150,
    validation_fraction=0.2,
    early_stopping=True,
    n_iter_no_change=25
)

hgb_clf.fit(train_X, train_y)

train_pred = hgb_clf.predict(train_X)
test_pred = hgb_clf.predict(test_X)

train_proba = hgb_clf.predict_proba(train_X)
test_proba = hgb_clf.predict_proba(test_X)

train_result = evaluate_predictions(
    y_true=train_y,
    y_pred=train_pred,
    y_proba=train_proba,
    classes=hgb_clf.classes_,
    dataset_name="Train"
)

test_result = evaluate_predictions(
    y_true=test_y,
    y_pred=test_pred,
    y_proba=test_proba,
    classes=hgb_clf.classes_,
    dataset_name="Test"
)

print_report(train_result)
print_report(test_result)

========== Train ==========
target type: binary
   accuracy: 0.996577
     recall: 0.987247
         f1: 0.989474
    roc_auc: 0.999533
     pr_auc: 0.998071
========== Test ==========
target type: binary
   accuracy: 0.985826
     recall: 0.936937
         f1: 0.955590
    roc_auc: 0.997870
     pr_auc: 0.990096
